# Versioning datasets alongside code

A water utility ranks its pipes by leak risk, and a crew inspects the 10 pipes the model ranks highest. In March
a colleague "cleaned" last year's pipe-repair log in place, and the retrained model caught fewer leaks. In this
notebook you make the same two versions of the repair log, find what the cleaning changed, build `snapshot()`
from a hash and a cache, then run the real DVC session from the lesson: `dvc add`, `dvc status`, the broken
`git checkout`, and the fix with `dvc checkout`.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and
no key. The utility and its data are made up; the training and the DVC commands are real.

## 1. Install DVC

The next cell installs the DVC version the lesson ran (3.67.1). Colab already has git. It also defines
`sh()`, which runs a shell command and prints it after a `$`, then its output, like a terminal. Every DVC and
git command in this notebook goes through it.

**Look for:** two version lines, `3.67.1` for DVC and a git version. pip may print a few lines about other
packages; you can ignore them.

In [1]:
%pip install -q dvc==3.67.1

import os, subprocess
def sh(cmd):
    # run a shell command; print it after a $, then its output, like a terminal
    print("$ " + cmd)
    env = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
    env["TERM"] = "dumb"  # plain text: no colours, no progress spinners
    out = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env).stdout
    print(out)

sh('dvc --version')
sh('git --version')

Note: you may need to restart the kernel to use updated packages.
$ dvc --version


3.67.1

$ git --version
git version 2.45.1



## 2. Make the two versions of the repair log

This is the lesson's data generator (seed 4). It writes three files into `gen/`:

- `pipes.csv`: one row per pipe: material, age, train or test, and whether it leaked in 2026.
- `pipe_repairs_jan.csv`: the 2025 repair log as it was in January, one row per repair job.
- `pipe_repairs_mar.csv`: the same log after the March "cleaning". A job was deleted when the same pipe already
  had a job in the 7 days before, and one typo in a note was fixed.

**Look for:** 180 pipes, 123 repair rows in January and 114 in March, 15 leaks among the test pipes.

In [2]:
import csv, os, shutil
from datetime import date, timedelta
import numpy as np

os.environ["DVC_NO_ANALYTICS"] = "1"
if "START" not in globals():
    START = os.getcwd()          # the folder this notebook started in
os.chdir(START)
shutil.rmtree("dvc-lesson", ignore_errors=True)
os.makedirs("dvc-lesson/gen")
os.makedirs("dvc-lesson/leak-model/data")
os.chdir("dvc-lesson")
WORK = os.getcwd()

rng = np.random.default_rng(4)
N = 180
materials = ["cast iron", "ductile iron", "PVC"]
pipes = []
for i in range(N):
    pid = f"P-{i + 1:03d}"
    mat = materials[rng.choice(3, p=[0.35, 0.35, 0.30])]
    age = int(rng.integers(5, 70))
    # hidden fragility: old cast iron is worse; a few pipes are badly broken ("come back" pipes)
    frag = 0.4 + 0.015 * age + (0.6 if mat == "cast iron" else 0.0) + rng.normal(0, 0.3)
    comeback = rng.random() < 0.06
    pipes.append(dict(pipe_id=pid, material=mat, age_years=age, frag=max(frag, 0.05), comeback=comeback))

repairs = []
start = date(2025, 1, 1)
for p in pipes:
    n = rng.poisson(p["frag"] * 0.6)
    for _ in range(n):
        d = start + timedelta(days=int(rng.integers(0, 365)))
        repairs.append([p["pipe_id"], d, "joint leak" if rng.random() < 0.5 else "main break", False])
    if p["comeback"]:
        # a repair that failed again within a few days: two jobs in one week
        d = start + timedelta(days=int(rng.integers(0, 350)))
        repairs.append([p["pipe_id"], d, "main break", False])
        for k in range(int(rng.integers(1, 2))):
            repairs.append([p["pipe_id"], d + timedelta(days=int(rng.integers(1, 6)) + k), "main break, same spot", True])

repairs.sort(key=lambda r: (r[1], r[0]))
rows = []
for j, (pid, d, note, rep) in enumerate(repairs):
    rows.append(dict(job_id=f"J-{j + 1:04d}", pipe_id=pid, date=d.isoformat(), note=note))

# labels for 2026: pipes whose repairs came back are the ones most likely to leak again
for p in pipes:
    z = -3.2 + 0.9 * p["frag"] + (2.6 if p["comeback"] else 0.0) + rng.normal(0, 0.5)
    p["leaked_2026"] = int(rng.random() < 1 / (1 + np.exp(-z)))
    p["split"] = "test" if int(p["pipe_id"][2:]) % 3 == 0 else "train"

# a typo in one note, present in January
typo_at = next(i for i, r in enumerate(rows) if r["note"] == "joint leak")
rows[typo_at]["note"] = "joint lek"

def write(path, rs, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, lineterminator="\n")
        w.writeheader()
        for r in rs:
            w.writerow({k: r[k] for k in fields})

write("gen/pipes.csv", pipes, ["pipe_id", "material", "age_years", "split", "leaked_2026"])
fields = ["job_id", "pipe_id", "date", "note"]
write("gen/pipe_repairs_jan.csv", rows, fields)

# March "cleaning": drop a job if the same pipe already had a job in the 7 days before; fix the typo
cleaned, last = [], {}
for r in rows:
    d = date.fromisoformat(r["date"])
    prev = last.get(r["pipe_id"])
    last[r["pipe_id"]] = d
    if prev is not None and (d - prev).days <= 7:
        continue
    r2 = dict(r)
    if r2["note"] == "joint lek":
        r2["note"] = "joint leak"
    cleaned.append(r2)
write("gen/pipe_repairs_mar.csv", cleaned, fields)
print("pipes:", N)
print("repair rows in january:", len(rows))
print("repair rows in march:  ", len(cleaned))
print("test pipes:", sum(p["split"] == "test" for p in pipes),
      "| test pipes that leaked in 2026:", sum(p["leaked_2026"] for p in pipes if p["split"] == "test"))

pipes: 180
repair rows in january: 123
repair rows in march:   114
test pipes: 60 | test pipes that leaked in 2026: 15


## 3. Find what the cleaning changed

`diff` prints every line that differs between the two files. `<` is a January line, `>` is a March line.
Then we print every row of four pipes in both versions and count their repairs. The model reads only the
repair count per pipe, the age and the material. It never reads the note.

**Look for:** one changed line (`joint lek` became `joint leak`) and 9 deleted rows, most of them
"main break, same spot". P-054 and P-177 go from 2 repairs to 1. P-102 keeps 3 and P-109 keeps 1.

In [3]:
sh('diff gen/pipe_repairs_jan.csv gen/pipe_repairs_mar.csv')

SHOW = ["P-054", "P-102", "P-109", "P-177"]
for name in ["jan", "mar"]:
    rs = list(csv.DictReader(open(f"gen/pipe_repairs_{name}.csv")))
    print(f"\n== {name}: {len(rs)} rows; rows for {SHOW}")
    for r in rs:
        if r["pipe_id"] in SHOW:
            print(",".join(r.values()))
    for pid in SHOW:
        print(pid, "repairs:", sum(r["pipe_id"] == pid for r in rs))

$ diff gen/pipe_repairs_jan.csv gen/pipe_repairs_mar.csv
2c2
< J-0001,P-109,2025-01-04,joint lek
---
> J-0001,P-109,2025-01-04,joint leak
33d32
< J-0032,P-054,2025-04-17,"main break, same spot"
45d43
< J-0044,P-111,2025-05-13,"main break, same spot"
47d44
< J-0046,P-096,2025-05-16,"main break, same spot"
53d49
< J-0052,P-165,2025-05-30,main break
87d82
< J-0086,P-052,2025-08-31,"main break, same spot"
94d88
< J-0093,P-108,2025-09-28,"main break, same spot"
102d95
< J-0101,P-081,2025-10-16,"main break, same spot"
120d112
< J-0119,P-177,2025-12-15,"main break, same spot"
122d113
< J-0121,P-116,2025-12-17,"main break, same spot"


== jan: 123 rows; rows for ['P-054', 'P-102', 'P-109', 'P-177']
J-0001,P-109,2025-01-04,joint lek
J-0030,P-054,2025-04-12,main break
J-0032,P-054,2025-04-17,main break, same spot
J-0084,P-102,2025-08-28,joint leak
J-0091,P-102,2025-09-16,joint leak
J-0110,P-102,2025-11-18,joint leak
J-0117,P-177,2025-12-12,main break
J-0119,P-177,2025-12-15,main break, same spot

## 4. A version of data is its bytes

MD5 turns a file's bytes into a 32-character code (a hash). We hash the January file and four variants of it.

**Look for:** the copy with a new name has the same MD5 as January (`ae670423...`). The typo fix (one byte
longer), the same rows sorted by pipe (same size), and the March file all get a completely different MD5.

In [4]:
import hashlib
md5 = lambda b: hashlib.md5(b).hexdigest()

jan = open("gen/pipe_repairs_jan.csv", "rb").read()
mar = open("gen/pipe_repairs_mar.csv", "rb").read()
shutil.copyfile("gen/pipe_repairs_jan.csv", "gen/repairs_2025_FINAL.csv")
renamed = open("gen/repairs_2025_FINAL.csv", "rb").read()
typo_fixed = jan.replace(b"joint lek", b"joint leak", 1)
header, *lines = jan.decode().splitlines(keepends=True)
by_pipe = (header + "".join(sorted(lines, key=lambda r: r.split(",")[1]))).encode()

print("january file          ", md5(jan), len(jan), "bytes")
print("same bytes, new name  ", md5(renamed), len(renamed), "bytes")
print("only the typo fixed   ", md5(typo_fixed), len(typo_fixed), "bytes")
print("rows sorted by pipe   ", md5(by_pipe), len(by_pipe), "bytes")
print("march file            ", md5(mar), len(mar), "bytes")

january file           ae67042367971abc8772461e3875050b 4433 bytes
same bytes, new name   ae67042367971abc8772461e3875050b 4433 bytes
only the typo fixed    2deac0456907dbe8025500b18948614e 4434 bytes
rows sorted by pipe    0ed4e5d4532f1b8ca5140854f7f59088 4433 bytes
march file             fd08e0c826c7eff502b30668fd33bd39 4015 bytes


## 5. Snapshots from scratch

`snapshot()` hashes the file, stores a copy in `cache/` under its hash (first 2 characters as a folder, the
rest as the file name), and writes the hash into a small pointer file next to the data. That pointer is what
git would commit. `restore()` goes back: it reads the pointer and copies the cache file over the data file.
We snapshot the January file, then the March file, then the January file again.

**Look for:** the cache holds 1 file, then 2, and still 2 after the third call: the same bytes are stored once.
After the restore, the file's MD5 changes from `fd08e0c8...` (March) back to `ae670423...` (January).

In [5]:
def snapshot(data_path, cache_dir):
    content = open(data_path, "rb").read()
    digest = hashlib.md5(content).hexdigest()
    stored = os.path.join(cache_dir, digest[:2], digest[2:])
    if not os.path.exists(stored):
        os.makedirs(os.path.dirname(stored), exist_ok=True)
        shutil.copyfile(data_path, stored)
    with open(data_path + ".ptr", "w") as pointer:
        pointer.write(digest + "\n")
    return digest

def restore(data_path, cache_dir):
    digest = open(data_path + ".ptr").read().strip()
    shutil.copyfile(os.path.join(cache_dir, digest[:2], digest[2:]), data_path)

os.makedirs("scratch/data", exist_ok=True)
os.chdir("scratch")
md5_of = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
listing = lambda: sorted(os.path.join(d, f) for d, _, fs in os.walk("cache") for f in fs)

for label, src in [("january", "pipe_repairs_jan.csv"), ("march", "pipe_repairs_mar.csv"), ("january again", "pipe_repairs_jan.csv")]:
    shutil.copyfile(os.path.join(WORK, "gen", src), "data/pipe_repairs.csv")
    result = snapshot("data/pipe_repairs.csv", "cache")
    print(f"== snapshot of the {label} file -> {result}")
    print("cache now holds:", listing())
    print("pointer file says:", open("data/pipe_repairs.csv.ptr").read().strip())

# go back: write January's pointer, then restore the file from the cache
shutil.copyfile(os.path.join(WORK, "gen", "pipe_repairs_mar.csv"), "data/pipe_repairs.csv")
open("data/pipe_repairs.csv.ptr", "w").write("ae67042367971abc8772461e3875050b\n")
print("before restore, file md5:", md5_of("data/pipe_repairs.csv"))
restore("data/pipe_repairs.csv", "cache")
print("after restore, file md5: ", md5_of("data/pipe_repairs.csv"))
os.chdir(WORK)

== snapshot of the january file -> ae67042367971abc8772461e3875050b
cache now holds: ['cache/ae/67042367971abc8772461e3875050b']
pointer file says: ae67042367971abc8772461e3875050b
== snapshot of the march file -> fd08e0c826c7eff502b30668fd33bd39
cache now holds: ['cache/ae/67042367971abc8772461e3875050b', 'cache/fd/08e0c826c7eff502b30668fd33bd39']
pointer file says: fd08e0c826c7eff502b30668fd33bd39
== snapshot of the january again file -> ae67042367971abc8772461e3875050b
cache now holds: ['cache/ae/67042367971abc8772461e3875050b', 'cache/fd/08e0c826c7eff502b30668fd33bd39']
pointer file says: ae67042367971abc8772461e3875050b
before restore, file md5: fd08e0c826c7eff502b30668fd33bd39
after restore, file md5:  ae67042367971abc8772461e3875050b


## 6. The leak model

From here on we work in a real project folder, `leak-model/`, the way the lesson's session did. The next cell
writes the lesson's `train.py` into it: a small logistic regression in NumPy with no randomness (the weights
start at zero). It counts repair jobs per pipe from the repair log, trains on the train pipes, ranks the test
pipes by risk, and counts how many of the top 10 really leaked. It also prints the MD5 of the file it read.

**Look for:** `Writing leak-model/train.py`. Nothing else runs yet.

In [6]:
%%writefile leak-model/train.py
"""The leak model for s6-data-versioning: logistic regression in NumPy, no randomness (weights start at 0).

python train.py <pipe_repairs.csv> <pipes.csv>
Features per pipe: age in decades, cast iron (0/1), repair jobs in 2025 (counted from the repair log).
Trained on the 'train' pipes. Score: the crew can inspect 10 pipes, so we rank the 'test' pipes by predicted
risk and count how many of the 10 highest really leaked in 2026 ("leaks caught in the top 10").
"""
import csv, sys, hashlib
import numpy as np

repairs_path, pipes_path = sys.argv[1], sys.argv[2]
with open(repairs_path, "rb") as f:
    print("data md5:", hashlib.md5(f.read()).hexdigest())

repairs_per_pipe = {}
with open(repairs_path) as f:
    for r in csv.DictReader(f):
        repairs_per_pipe[r["pipe_id"]] = repairs_per_pipe.get(r["pipe_id"], 0) + 1

pipes = list(csv.DictReader(open(pipes_path)))
X = np.array([[1.0, int(p["age_years"]) / 10, p["material"] == "cast iron", repairs_per_pipe.get(p["pipe_id"], 0)] for p in pipes], dtype=float)
y = np.array([int(p["leaked_2026"]) for p in pipes], dtype=float)
train = np.array([p["split"] == "train" for p in pipes])
test = ~train

w = np.zeros(X.shape[1])
for _ in range(3000):
    p = 1 / (1 + np.exp(-X[train] @ w))
    w -= 0.05 * X[train].T @ (p - y[train]) / train.sum()

risk = 1 / (1 + np.exp(-X[test] @ w))
top = np.argsort(-risk, kind="stable")[:10]
ids = [pipes[i]["pipe_id"] for i in np.flatnonzero(test)[top]]
print("weights (bias, age/10, cast iron, repairs):", np.round(w, 2).tolist())
print("top 10 test pipes:", ids)
print("leaks caught in the top 10:", int(y[test][top].sum()), "of", int(y[test].sum()), "test leaks")

Writing leak-model/train.py


## 7. January: put the repair log under DVC

We make a git repository, copy in the pipe table and the January repair log, and run `dvc init` and
`dvc add`. DVC hashes the file, stores it in its cache `.dvc/cache`, writes the small pointer file
`data/pipe_repairs.csv.dvc`, and adds the data file to `data/.gitignore` so git skips it.

**Look for:** the `.dvc` file shows `md5: ae67042367971abc8772461e3875050b` and `size: 4433`, the same hash
as your January file in step 4. `data/.gitignore` contains `/pipe_repairs.csv`.

In [7]:
os.chdir("leak-model")
sh('git init -q -b main')
sh('git config user.name "Leak team" && git config user.email "leak-team@example.com"')
shutil.copyfile(os.path.join(WORK, "gen", "pipes.csv"), "data/pipes.csv")
shutil.copyfile(os.path.join(WORK, "gen", "pipe_repairs_jan.csv"), "data/pipe_repairs.csv")

sh('dvc init -q')
sh('dvc add data/pipe_repairs.csv')
sh('cat data/pipe_repairs.csv.dvc')
sh('cat data/.gitignore')

$ git init -q -b main



$ git config user.name "Leak team" && git config user.email "leak-team@example.com"



$ dvc init -q



$ dvc add data/pipe_repairs.csv



To track the changes with git, run:

	git add data/.gitignore data/pipe_repairs.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

$ cat data/pipe_repairs.csv.dvc
outs:
- md5: ae67042367971abc8772461e3875050b
  size: 4433
  hash: md5
  path: pipe_repairs.csv

$ cat data/.gitignore
/pipe_repairs.csv



## 8. Commit the January model and train it

Git commits the code and the pointer, never the data file. The tag `jan-model` marks this commit.

**Look for:** `data md5: ae670423...` and `leaks caught in the top 10: 7 of 15 test leaks`.

In [8]:
sh('git add .dvc .dvcignore train.py data/pipes.csv data/pipe_repairs.csv.dvc data/.gitignore')
sh('git commit -q -m "January model: code + data pointer"')
sh('git tag jan-model')
sh('python train.py data/pipe_repairs.csv data/pipes.csv')

$ git add .dvc .dvcignore train.py data/pipes.csv data/pipe_repairs.csv.dvc data/.gitignore

$ git commit -q -m "January model: code + data pointer"

$ git tag jan-model



$ python train.py data/pipe_repairs.csv data/pipes.csv


data md5: ae67042367971abc8772461e3875050b
weights (bias, age/10, cast iron, repairs): [-2.54, -0.02, 0.15, 0.66]
top 10 test pipes: ['P-102', 'P-171', 'P-081', 'P-096', 'P-111', 'P-165', 'P-075', 'P-177', 'P-054', 'P-126']
leaks caught in the top 10: 7 of 15 test leaks



## 9. March: the colleague cleans the file in place

We overwrite the repair log with the March version, keeping its name. `dvc status` hashes the file in the
folder and compares it with the md5 in the pointer. Then `dvc add` stores the new version, and git sees the
pointer change.

**Look for:** `modified: data/pipe_repairs.csv`, then a `git diff` where the md5 changes to
`fd08e0c826c7eff502b30668fd33bd39` and the size to `4015`, then `leaks caught in the top 10: 4 of 15`.
Last, the cell lists the DVC cache. DVC stores each version whole, under its hash, the same layout as your
`cache/` in step 5: 2 files, in folders `ae` and `fd`, of 4433 and 4015 bytes.

In [9]:
shutil.copyfile(os.path.join(WORK, "gen", "pipe_repairs_mar.csv"), "data/pipe_repairs.csv")
sh('dvc status')
sh('dvc add data/pipe_repairs.csv')
sh('git diff data/pipe_repairs.csv.dvc')
sh('git commit -q -am "Cleaned repair log"')
sh('python train.py data/pipe_repairs.csv data/pipes.csv')

print("\nfiles in the DVC cache:")
for d, _, fs in sorted(os.walk(".dvc/cache/files")):
    for f in fs:
        path = os.path.join(d, f)
        print(path, os.path.getsize(path), "bytes")

$ dvc status


data/pipe_repairs.csv.dvc:
	changed outs:
		modified:           data/pipe_repairs.csv

$ dvc add data/pipe_repairs.csv



To track the changes with git, run:

	git add data/pipe_repairs.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

$ git diff data/pipe_repairs.csv.dvc
diff --git a/data/pipe_repairs.csv.dvc b/data/pipe_repairs.csv.dvc
index 98d8e04..131bff9 100644
--- a/data/pipe_repairs.csv.dvc
+++ b/data/pipe_repairs.csv.dvc
@@ -1,5 +1,5 @@
 outs:
-- md5: ae67042367971abc8772461e3875050b
-  size: 4433
+- md5: fd08e0c826c7eff502b30668fd33bd39
+  size: 4015
   hash: md5
   path: pipe_repairs.csv

$ git commit -q -am "Cleaned repair log"

$ python train.py data/pipe_repairs.csv data/pipes.csv


data md5: fd08e0c826c7eff502b30668fd33bd39
weights (bias, age/10, cast iron, repairs): [-2.47, -0.02, 0.15, 0.6]
top 10 test pipes: ['P-102', 'P-171', 'P-075', 'P-126', 'P-120', 'P-063', 'P-048', 'P-123', 'P-081', 'P-096']
leaks caught in the top 10: 4 of 15 test leaks


files in the DVC cache:
.dvc/cache/files/md5/ae/67042367971abc8772461e3875050b 4433 bytes
.dvc/cache/files/md5/fd/08e0c826c7eff502b30668fd33bd39 4015 bytes


## 10. Broken on purpose: back to January with git only

A month later, the team wants the January model back. A teammate checks out the January tag and retrains.
(`-c advice.detachedHead=false` only silences a git notice.)

**Look for:** the pointer asks for `ae670423...`, but `train.py` prints `data md5: fd08e0c8...` (the March
file) and still catches 4 leaks. `dvc status` says the file does not match the pointer. Git changed the
`.dvc` file and left the data alone, because the data is in `.gitignore`.

In [10]:
sh('git -c advice.detachedHead=false checkout -q jan-model')
sh('cat data/pipe_repairs.csv.dvc')
sh('python train.py data/pipe_repairs.csv data/pipes.csv')
sh('dvc status')

$ git -c advice.detachedHead=false checkout -q jan-model

$ cat data/pipe_repairs.csv.dvc
outs:
- md5: ae67042367971abc8772461e3875050b
  size: 4433
  hash: md5
  path: pipe_repairs.csv

$ python train.py data/pipe_repairs.csv data/pipes.csv


data md5: fd08e0c826c7eff502b30668fd33bd39
weights (bias, age/10, cast iron, repairs): [-2.47, -0.02, 0.15, 0.6]
top 10 test pipes: ['P-102', 'P-171', 'P-075', 'P-126', 'P-120', 'P-063', 'P-048', 'P-123', 'P-081', 'P-096']
leaks caught in the top 10: 4 of 15 test leaks

$ dvc status


data/pipe_repairs.csv.dvc:
	changed outs:
		modified:           data/pipe_repairs.csv



## 11. The fix: dvc checkout

`dvc checkout` reads every `.dvc` file and makes the folder match it, copying the right version from the cache.

**Look for:** `M       data/pipe_repairs.csv`, then `data md5: ae670423...` and `7 of 15 test leaks`, then
`Data and pipelines are up to date.`

In [11]:
sh('dvc checkout')
sh('python train.py data/pipe_repairs.csv data/pipes.csv')
sh('dvc status')

$ dvc checkout


M       data/pipe_repairs.csv

$ python train.py data/pipe_repairs.csv data/pipes.csv


data md5: ae67042367971abc8772461e3875050b
weights (bias, age/10, cast iron, repairs): [-2.54, -0.02, 0.15, 0.66]
top 10 test pipes: ['P-102', 'P-171', 'P-081', 'P-096', 'P-111', 'P-165', 'P-075', 'P-177', 'P-054', 'P-126']
leaks caught in the top 10: 7 of 15 test leaks

$ dvc status


Data and pipelines are up to date.



## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it and run the cell again.

### Experiment 1: change one byte, see a new version

The cell writes a variant of the January file and snapshots it with your `snapshot()` from step 5.

**What should change:** as written, the variant is the January file itself, so its MD5 is `ae670423...` and the
cache still holds 2 files. Change the line to `jan + b"\n"` (one extra empty line at the end). The file is one
byte longer, the MD5 is completely different, and the cache grows to 3 files: any change is a new whole copy.

In [12]:
variant = jan  # <- change this (try: jan + b"\n")

os.chdir(os.path.join(WORK, "scratch"))
open("data/variant.csv", "wb").write(variant)
print("variant md5:", snapshot("data/variant.csv", "cache"), len(variant), "bytes")
print("cache holds", len(listing()), "files")
os.chdir(os.path.join(WORK, "leak-model"))

variant md5: ae67042367971abc8772461e3875050b 4433 bytes
cache holds 2 files


### Experiment 2: a gentler cleaning

The March cleaning deleted a job when the same pipe had another job in the 7 days before. Try a smaller window.
The cell cleans the January file again with your window and trains the model on the result (outside DVC).

**What should change:** with 7 days you get the March file again: 114 rows, `fd08e0c8...`, 4 leaks caught.
With `WINDOW_DAYS = 0` no row is deleted (123 rows) and only the typo is fixed: the MD5 is `2deac045...`, the
same as the typo-fixed variant in step 4, and the model catches 7 again, because it never reads the note. With
3 days, 118 rows are kept and the score is back to 4: deleting even a few repeat repairs loses the leaks.

In [13]:
WINDOW_DAYS = 7  # <- change this (try 0, then 3)

kept, last = [], {}
for r in csv.DictReader(open(os.path.join(WORK, "gen", "pipe_repairs_jan.csv"))):
    d = date.fromisoformat(r["date"])
    prev = last.get(r["pipe_id"])
    last[r["pipe_id"]] = d
    if prev is not None and (d - prev).days <= WINDOW_DAYS:
        continue
    if r["note"] == "joint lek":
        r["note"] = "joint leak"
    kept.append(r)
write(os.path.join(WORK, "my_cleaning.csv"), kept, fields)
print("rows kept:", len(kept))
sh('python train.py ../my_cleaning.csv data/pipes.csv')

rows kept: 114
$ python train.py ../my_cleaning.csv data/pipes.csv


data md5: fd08e0c826c7eff502b30668fd33bd39
weights (bias, age/10, cast iron, repairs): [-2.47, -0.02, 0.15, 0.6]
top 10 test pipes: ['P-102', 'P-171', 'P-075', 'P-126', 'P-120', 'P-063', 'P-048', 'P-123', 'P-081', 'P-096']
leaks caught in the top 10: 4 of 15 test leaks



### Experiment 3: go to any commit with its data

The cell checks out a commit, then runs `dvc checkout` so the data follows the pointer, and trains.

**What should change:** as written, `TARGET` is the January tag: `ae670423...` and 7 leaks. Change it to
`"main"`, the branch with the March commit: the data file follows to `fd08e0c8...` and 4 leaks. Remove the
`dvc checkout` line and run again with the other target: the data stays behind, as in step 10.

In [14]:
TARGET = "jan-model"  # <- change this (try "main")

sh(f'git -c advice.detachedHead=false checkout -q {TARGET}')
sh('dvc checkout')
sh('python train.py data/pipe_repairs.csv data/pipes.csv')

$ git -c advice.detachedHead=false checkout -q jan-model

$ dvc checkout



$ python train.py data/pipe_repairs.csv data/pipes.csv


data md5: ae67042367971abc8772461e3875050b
weights (bias, age/10, cast iron, repairs): [-2.54, -0.02, 0.15, 0.66]
top 10 test pipes: ['P-102', 'P-171', 'P-081', 'P-096', 'P-111', 'P-165', 'P-075', 'P-177', 'P-054', 'P-126']
leaks caught in the top 10: 7 of 15 test leaks



## What you should see

- Step 7: after `dvc add data/pipe_repairs.csv`, `data/pipe_repairs.csv.dvc` shows an `md5` line and
  `size: 4433`, and `data/.gitignore` contains `/pipe_repairs.csv`.
- Step 9: after you overwrite the file with the cleaned version, `dvc status` reports
  `modified: data/pipe_repairs.csv`.
- Step 9: after the second `dvc add`, `.dvc/cache/files/md5` holds 2 files, in folders named by the first two
  characters of each hash.
- Steps 10 and 11: after `git checkout jan-model` alone, `train.py` prints `data md5: fd08e0c8...` and 4 leaks
  caught; after `dvc checkout` it prints `ae670423...` and 7.